In [8]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [2]:
column_names = [
    "CrimeRate",  # The per capita crime rate by town. lower housing values
    "ResidentialLandZoned", # The proportion of residential land zoned for lots over 25,000 sq.ft.
    "IndustrialLandUse", # The proportion of non-retail business acres per town.
    "CharlesRiver", # Charles River dummy variable (1 if tract bounds river; 0 otherwise)
    "NitrixOxide", # Nitric oxides concentration (parts per 10 million)
    "AvgRoomPerDwelling", # The average number of rooms per dwelling
    "HousingAge", # The proportion of owner-occupied units built prior to 1940
    "DistanceToWork", # Weighted distances to five Boston employment centres
    "HighwayAccess", # Index of accessibility to radial highways
    "PropertyTaxRate", # Full-value property-tax rate per $10,000
    "Pupil-TeacherRatio", # The pupil-teacher ratio by town
    "B", # 1000(Bk - 0.63)^2 where Bk is the proportion of blacks by town
    "LowSocioEcomic", # The percentage of lower status of the population
    "Value", # Median value of owner-occupied homes in $1000s
]
df = pd.read_csv("data/housing.csv", sep=r"\s+", header=None, names=column_names)

In [3]:
df.head(2)

,CrimeRate,ResidentialLandZoned,IndustrialLandUse,CharlesRiver,NitrixOxide,AvgRoomPerDwelling,HousingAge,DistanceToWork,HighwayAccess,PropertyTaxRate,Pupil-TeacherRatio,B,LowSocioEcomic,Value
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.9,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.9,9.14,21.6


In [1]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import mlflow
from mlflow.entities import ViewType
from mlflow.tracking import MlflowClient
client = MlflowClient()

mlflow.set_tracking_uri("http://localhost:5020")
mlflow.set_experiment("Boston Model Experiment")
# mlflow.set_registry_uri("sqlite:///mlflow_registry.db")

<Experiment: artifact_location='mlflow-artifacts:/417350048653659120', creation_time=1756917308481, experiment_id='417350048653659120', last_update_time=1756917308481, lifecycle_stage='active', name='Boston Model Experiment', tags={}>

In [6]:
feature_cols = ["CrimeRate", "ResidentialLandZoned", "IndustrialLandUse", "CharlesRiver", "NitrixOxide", "AvgRoomPerDwelling", "HousingAge", "DistanceToWork", "HighwayAccess", "PropertyTaxRate", "Pupil-TeacherRatio", "B", "LowSocioEcomic"]
X = df[feature_cols]
y = df['Value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
mlflow.start_run(run_name="XGBRegressor Run")

xgb_model = XGBRegressor(n_estimators=150, max_depth=5, learning_rate=0.05, random_state=42)
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5

mlflow.log_metric("rmse", rmse)
mlflow.xgboost.log_model(xgb_model, "xgbr_model")

model_name = "XGBRegressor Model"
model_uri = f"runs:/{mlflow.active_run().info.run_id}/xgbr_model"
mlflow.register_model(model_uri, model_name)

mlflow.end_run()



2025/09/04 23:09:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\ProgramData\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [23:09:22] WARNING: C:\b\abs_90_bwj_86a\croot\xgboost-split_1724073762025\work\src\c_api\c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)
2025/09/04 23:09:27 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/09/04 23:09:30 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/09/04 23:09:30 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade  -> 451

🏃 View run XGBRegressor Run at: http://localhost:5020/#/experiments/417350048653659120/runs/c17cc64c833140929519e5e7ab80e20b
🧪 View experiment at: http://localhost:5020/#/experiments/417350048653659120


1. A logged model is a model artifact saved during an MLflow run. It is stored in the MLflow tracking server or a specified artifact storage location. Logged models are primarily used for experimentation and tracking purposes.
- Purpose: Captures the model, its parameters, metrics, and artifacts during training.
- Storage: Saved as part of an MLflow run under the artifacts section.
- Access: Can be loaded using the mlflow.<model_flavor>.load_model() method with a run-relative path (e.g., runs:/<run_id>/<model_path>).
- Use Case: Ideal for experimentation, debugging, and intermediate model evaluation.

2. A registered model is a model that has been added to the MLflow Model Registry. It provides a centralized system for managing models across their lifecycle, including versioning, staging, and deployment.

- Purpose: Enables collaborative model management with features like versioning, aliases, and metadata tagging.
- Storage: Stored in the Model Registry with a unique name and associated versions.
- Access: Can be loaded using a model URI (e.g., models:/<model_name>/<version> or models:/<model_name>@<alias>).
- Use Case: Suitable for production workflows, deployment, and governance.

In [ ]:
# Load registered model and make predictions
loaded_model = mlflow.pyfunc.load_model(model_uri)
loaded_model.predict(X_test)

In [2]:

# Exact name match
for exp in mlflow.search_experiments(
        view_type=ViewType.ALL,
        filter_string="name = 'Boston Model Experiment'"):
    print(exp.experiment_id, exp.name, exp.lifecycle_stage)

417350048653659120 Boston Model Experiment active


In [2]:
for exp in mlflow.search_experiments(
        filter_string="name LIKE '%Boston%'"):
    print(exp.experiment_id, exp.name)

417350048653659120 Boston Model Experiment


In [3]:
exp = mlflow.get_experiment_by_name("Boston Model Experiment")
if exp is None:
    raise ValueError(f"Experiment not found: Boston Model Experiment")
runs = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["start_time DESC"],
    max_results=3,
)
if runs.empty:
    raise RuntimeError("No finished runs found.")
run_id = runs.iloc[0]["run_id"]

In [4]:
runs

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.rmse,tags.mlflow.user,tags.mlflow.source.type,tags.mlflow.source.name,tags.mlflow.runName
0,c17cc64c833140929519e5e7ab80e20b,417350048653659120,FINISHED,mlflow-artifacts:/417350048653659120/c17cc64c8...,2025-09-05 06:09:20.038000+00:00,2025-09-05 06:09:30.612000+00:00,2.421340,mamma,LOCAL,c:\ProgramData\anaconda3\Lib\site-packages\ipy...,XGBRegressor Run
1,d2de8e82dfcf4bb1ac6dddcc5f3a74d9,417350048653659120,FINISHED,mlflow-artifacts:/417350048653659120/d2de8e82d...,2025-09-03 16:43:41.177000+00:00,2025-09-03 16:43:46.637000+00:00,2.659466,mamma,LOCAL,c:\ProgramData\anaconda3\Lib\site-packages\ipy...,XGBRegressor Run
2,5e803f75d4564e1abf300ce9fac37bfc,417350048653659120,FINISHED,mlflow-artifacts:/417350048653659120/5e803f75d...,2025-09-03 16:42:41.847000+00:00,2025-09-03 16:42:51.456000+00:00,2.659466,mamma,LOCAL,c:\ProgramData\anaconda3\Lib\site-packages\ipy...,XGBRegressor Run


In [ ]:
import mlflow
from mlflow.entities import ViewType
from mlflow.tracking import MlflowClient
client = MlflowClient()

mlflow.set_tracking_uri("http://localhost:5020")
mlflow.set_experiment("Boston Model Experiment")
from urllib.parse import urlparse, unquote
run = client.get_run(run_id)
model = mlflow.pyfunc.load_model(f"runs:/d2de8e82dfcf4bb1ac6dddcc5f3a74d9/model")
for a in client.list_artifacts(run_id):
    print("ARTIFACT:", a.path)
    art_uri = run.info.artifact_uri               # e.g., file:///C:/.../mlruns/<exp>/<run>/artifacts
p = urlparse(art_uri)
if p.scheme != "file":
    raise RuntimeError(f"Artifact store is not local file:// (got {art_uri})")